In [2]:
!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cpu
!pip install transformers datasets evaluate pandas tqdm

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu


In [3]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel, pipeline
from torch.optim import AdamW


/Users/tanishapriya/Desktop/Capstone/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# 1. Configuration
# -------------------------------------------------
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"   # Base model
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN = 512
BATCH_SIZE = 8
EPOCHS = 2
LR = 2e-5

print("Using device:", DEVICE)

Using device: cpu


In [5]:
#2. Dataset Loader
# -------------------------------------------------
class ClinicalNotesDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=512):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        text = str(self.data.iloc[index]["cleaned_note"])
        inputs = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        label = torch.tensor(0)  # dummy label
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "labels": label,
        }


In [6]:
#3. Model Definition
# -------------------------------------------------
class ClinicalBERTModel(nn.Module):
    def __init__(self, model_name=MODEL_NAME, num_labels=2):
        super(ClinicalBERTModel, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]  # CLS token
        pooled_output = self.dropout(pooled_output)
        return self.fc(pooled_output)

# -------------------------------------------------

In [9]:
#4. Training Loop
# -------------------------------------------------
def train_model():
    # Example CSV: data/processed/clinical_notes_cleaned.csv
    if not os.path.exists("data/processed/clinical_notes_cleaned.csv"):
        # create dummy data
        os.makedirs("data/processed", exist_ok=True)
        pd.DataFrame({
            "cleaned_note": [
                "Patient admitted with pneumonia and treated with antibiotics.",
                "65 year old male with diabetes and hypertension, started insulin therapy."
            ]
        }).to_csv("data/processed/clinical_notes_cleaned.csv", index=False)

    df = pd.read_csv("data/processed/clinical_notes_cleaned.csv")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    dataset = ClinicalNotesDataset(df, tokenizer, MAX_LEN)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    model = ClinicalBERTModel(MODEL_NAME).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(EPOCHS):
        loop = tqdm(dataloader, leave=True)
        for batch in loop:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)

            labels = labels.view(-1)
            if logits.size(0) != labels.size(0):
                logits = logits[: labels.size(0), :]

            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            loop.set_description(f"Epoch {epoch+1}")
            loop.set_postfix(loss=loss.item())

    os.makedirs("models", exist_ok=True)
    torch.save(model.state_dict(), "models/clinical_bert.pth")
    print("✅ Training complete. Model saved to models/clinical_bert.pth")


In [10]:
def evaluate_model():
    import evaluate
    rouge = evaluate.load("rouge")

    preds = [
        "Patient treated with antibiotics for pneumonia.",
        "Male patient started insulin for diabetes."
    ]
    refs = [
        "Pneumonia treated with antibiotics.",
        "Diabetes patient received insulin therapy."
    ]
    results = rouge.compute(predictions=preds, references=refs)
    print("\n📊 ROUGE Results:", results)


In [11]:
 #6. Summarization Pipeline (T5 or BART)
# -------------------------------------------------
def run_summarizer():
    summarizer = pipeline("summarization", model="t5-small")
    text = """
    Patient is a 65-year-old male admitted after a fall at home.
    Presented with a right hip fracture and mild head trauma.
    Underwent successful open reduction and internal fixation of the hip.
    Post-op vitals stable, pain controlled with acetaminophen.
    Physical therapy initiated on day 2 post-op.
    """
    summary = summarizer(text, max_length=60, min_length=20, do_sample=False)
    print("\n📝 Clinical Note Summary:\n", summary[0]['summary_text'])

# -------------------------------------------------
# 7. Run Steps
# -------------------------------------------------
if __name__ == "__main__":
    train_model()
    evaluate_model()
    run_summarizer()

Epoch 2: 100%|██████████| 1/1 [00:03<00:00,  3.46s/it, loss=0.455]


✅ Training complete. Model saved to models/clinical_bert.pth

📊 ROUGE Results: {'rouge1': np.float64(0.6727272727272727), 'rouge2': np.float64(0.25), 'rougeL': np.float64(0.4818181818181818), 'rougeLsum': np.float64(0.4818181818181818)}


Device set to use mps:0
Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📝 Clinical Note Summary:
 patient is a 65-year-old male admitted after a fall at home . underwent successful open reduction and internal fixation of the hip .


In [12]:
import torch
from transformers import AutoTokenizer, AutoModel

# Load Bio_ClinicalBERT
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

# Example clinical note
text = """Patient is a 65-year-old male admitted after a fall at home. 
Presented with a right hip fracture and mild head trauma. 
Underwent successful open reduction and internal fixation of the hip. 
Post-op vitals stable, pain controlled with acetaminophen. 
No neurological deficits observed. Physical therapy initiated on day 2 post-op."""

# Tokenize
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)

# CLS embedding (representation of the whole note)
cls_embedding = outputs.last_hidden_state[:, 0, :]
print("CLS embedding shape:", cls_embedding.shape)

CLS embedding shape: torch.Size([1, 768])


In [14]:
!pip install scikit-learn

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 1.2 MB/s  0:00:07 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 994.4 kB/s  0:00:20:00:010:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]


In [21]:
texts = [
    "Patient is a 65-year-old male admitted after a fall at home with right hip fracture.",
    "The patient has a history of hypertension and diabetes, currently on insulin therapy.",
    "CT scan shows mild traumatic brain injury, patient is stable and under observation.",
    "Post-operative recovery is good, pain managed with acetaminophen and physiotherapy.",
]

# Tokenize all
inputs = tokenizer(texts, return_tensors="pt", truncation=True, padding=True, max_length=512)

with torch.no_grad():
    outputs = model(**inputs)

embeddings = outputs.last_hidden_state[:, 0, :].numpy()  # CLS embeddings
print("Embeddings shape:", embeddings.shape)  # (num_texts, hidden_dim)
embeddings = np.vstack([emb1, emb2, emb3])  # multiple CLS embeddings
sns.heatmap(embeddings, cmap="viridis")
plt.xlabel("Hidden Dimension")
plt.ylabel("Clinical Notes")

Embeddings shape: (4, 768)


NameError: name 'emb1' is not defined

In [18]:
embeddings = np.vstack([emb1, emb2, emb3])  # multiple CLS embeddings
sns.heatmap(embeddings, cmap="viridis")
plt.xlabel("Hidden Dimension")
plt.ylabel("Clinical Notes")

NameError: name 'emb1' is not defined

In [15]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import numpy as np

# Convert CLS embedding to numpy
embedding = cls_embedding.detach().cpu().numpy()

# --------------------------
# 1. PCA Visualization (2D)
# --------------------------
pca = PCA(n_components=2)
pca_result = pca.fit_transform(embedding)

plt.figure(figsize=(6,6))
plt.scatter(pca_result[:,0], pca_result[:,1], c='blue', alpha=0.7)
plt.title("PCA Visualization of Clinical Note Embedding")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.show()

# --------------------------
# 2. t-SNE Visualization (2D)
# --------------------------
tsne = TSNE(n_components=2, perplexity=5, random_state=42)
tsne_result = tsne.fit_transform(embedding)

plt.figure(figsize=(6,6))
plt.scatter(tsne_result[:,0], tsne_result[:,1], c='green', alpha=0.7)
plt.title("t-SNE Visualization of Clinical Note Embedding")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")
plt.show()

ValueError: n_components=2 must be between 0 and min(n_samples, n_features)=1 with svd_solver='full'